## 실습 1. 라이브러리 준비하기

In [1]:
import numpy as np

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

### 실행 결과 요약

- NumPy 준비: 수치 계산을 위해 `numpy`를 불러왔다.
- pandas 준비: 도서 데이터를 DataFrame 형태로 처리하기 위해 `pandas`를 불러왔다.
- TF-IDF 준비: 도서 제목을 숫자 벡터로 변환하기 위해 `TfidfVectorizer`를 불러왔다.
- 코사인 유사도 준비: 도서 제목 벡터 사이의 유사도를 계산하기 위해 `cosine_similarity`를 불러왔다.
- 결과 확인: 오류 없이 라이브러리가 불러와져 비슷한 도서를 추천하는 실습을 진행할 준비가 완료되었다.

## 실습 2. 데이터 불러오기

In [2]:
# 전처리된 도서 데이터 불러오기
DATA_PATH = "book_bestseller_clean.csv"

df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig"
)

# 데이터 구조 확인
print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())

# 추천에 사용할 주요 정보 확인
df_books[["상품명", "저자", "출판사", "분야"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명,저자,출판사,분야
0,소년이 온다,한강,창비,소설
1,모순,양귀자,쓰다,소설
2,결국 국민이 합니다,이재명,오마이북,정치/사회
3,혼모노,성해나,창비,소설
4,급류,정대건,민음사,소설
5,초역 부처의 말,코이케 류노스케,포레스트북스,인문
6,청춘의 독서(특별증보판),유시민,웅진지식하우스,인문
7,어른의 행복은 조용하다,태수,페이지2북스,시/에세이
8,채식주의자,한강,창비,소설
9,단 한 번의 삶(강물에디션 활판인쇄 한정판),김영하,복복서가,시/에세이


### 실행 결과 요약

- 데이터 크기 확인: 불러온 데이터는 총 **199행, 8개 컬럼**으로 구성되어 있다.
- 컬럼 확인: `순위`, `판매상품ID`, `상품명`, `판매가`, `저자`, `출판사`, `발행일`, `분야` 컬럼이 정상적으로 존재한다.
- 결측치 확인: 추천의 핵심 입력값인 `상품명` 컬럼의 결측치는 **0개**로 확인되었다.
- 추천 정보 확인: `상품명`, `저자`, `출판사`, `분야` 컬럼이 모두 정상적으로 출력되었다.
- 데이터 예시 확인: `소년이 온다`, `모순`, `결국 국민이 합니다` 등 실제 도서 제목과 저자, 출판사, 분야 정보가 함께 확인되었다.
- 결과 확인: 도서 제목을 기반으로 TF-IDF 벡터와 코사인 유사도를 계산하고, 추천 결과에 저자·출판사·분야 정보를 함께 표시할 수 있는 상태임을 확인했다.

## 실습3. 추천용 데이터 준비하기

In [3]:
# 추천용 데이터 복사
df_reco = df_books.copy()

# 상품명 결측치와 공백 정리
df_reco["상품명"] = (
    df_reco["상품명"]
    .fillna("")      # 결측치는 빈 문자열로 변경
    .astype(str)     # 문자열로 변환
    .str.strip()     # 앞뒤 공백 제거
)

# 빈 상품명 제거 후 인덱스 다시 정리
df_reco = (
    df_reco[df_reco["상품명"] != ""]
    .reset_index(drop=True)
)

# 추천에 사용할 도서 수 확인
print("추천에 사용할 도서 수:", len(df_reco))

# 인덱스와 상품명 확인
df_reco[["상품명"]].head()

추천에 사용할 도서 수: 199


,상품명
0,소년이 온다
1,모순
2,결국 국민이 합니다
3,혼모노
4,급류


### 실행 결과 요약

- 추천 도서 수 확인: 추천에 사용할 도서는 총 **199권**으로 확인되었다.
- 빈 제목 제거 확인: 상품명이 비어 있는 데이터가 없어 전체 199권이 그대로 추천 대상에 포함되었다.
- 인덱스 재정렬 확인: DataFrame 인덱스가 `0, 1, 2, 3, 4 ...` 순서로 정상적으로 정리되었다.
- 상품명 확인: `소년이 온다`, `모순`, `결국 국민이 합니다`, `혼모노`, `급류` 등이 정상적으로 출력되었다.
- 행 번호 대응 확인: 이후 TF-IDF 행렬의 각 행 번호와 `df_reco`의 인덱스를 같은 도서 기준으로 연결할 수 있는 상태가 되었다.
- 결과 확인: 총 **199권의 도서 데이터**가 추천 시스템에 사용할 수 있도록 정상적으로 준비되었다.

## 실습4. 콘텐츠 기반 추천 이해하기

### 콘텐츠 기반 추천 이해하기

- 정의: 콘텐츠 기반 추천은 선택한 항목의 특징과 비슷한 특징을 가진 다른 항목을 찾아 추천하는 방식이다.
- 사용 특징: 이번 실습에서는 도서의 `상품명` 텍스트를 도서의 특징으로 사용한다.
- 벡터화: 각 도서 제목을 TF-IDF 벡터로 변환한다.
- 유사도 비교: 두 도서의 TF-IDF 벡터가 얼마나 비슷한지 비교해 유사한 도서를 찾는다.
- 추천 방식: 제목의 텍스트 특징이 비슷할수록 추천 후보가 될 가능성이 높다.
- 사용하지 않는 정보: 사용자 구매 이력, 클릭 이력, 평점, 판매량, 개인 취향은 사용하지 않는다.
- 결과 해석: 따라서 이번 추천은 개인화 추천이 아니라 **도서 제목 기반 유사 도서 추천**이라고 이해하는 것이 정확하다.

도서 제목
    ↓
TF-IDF 벡터
    ↓
벡터 간 유사도 계산
    ↓
비슷한 제목의 도서 찾기
    ↓
추천

## 실습5. 코사인 유사도 이해하기

In [4]:
# 간단한 벡터 만들기
vectors = np.array([
    [1, 1],
    [2, 2],
    [1, 0]
])

# 코사인 유사도 계산
similarity_matrix = cosine_similarity(vectors)

print(similarity_matrix)

[[1.         1.         0.70710678]
 [1.         1.         0.70710678]
 [0.70710678 0.70710678 1.        ]]


### 코사인 유사도 이해하기

- 정의: 코사인 유사도는 두 벡터의 **방향이 얼마나 비슷한지**를 비교하는 값이다.
- 값의 의미: 값이 **1에 가까울수록 두 벡터가 매우 비슷한 방향**을 가진다.
- 값의 의미: 값이 **0에 가까울수록 두 벡터의 공통 특징이 적다**고 볼 수 있다.
- 크기와 방향: 벡터의 크기가 달라도 같은 방향을 가리키면 높은 유사도를 가질 수 있다.
- 자기 자신 비교: 하나의 벡터를 자기 자신과 비교하면 일반적으로 코사인 유사도는 **1**이 된다.
- 추천 활용: 도서 제목을 TF-IDF 벡터로 바꾼 뒤 코사인 유사도를 계산해 서로 비슷한 제목의 도서를 찾는다.
- 주의점: 자기 자신은 항상 가장 높은 유사도를 가지므로 실제 추천 결과에서는 자기 자신을 제외해야 한다.

## 실습 6. 실제 도서 제목을 TF-IDF로 변환하기

In [5]:
# 추천에 사용할 도서 제목
titles = df_reco["상품명"]

# TF-IDF Vectorizer 생성
tfidf = TfidfVectorizer()

# 전체 도서 제목을 TF-IDF 벡터로 변환
tfidf_matrix = tfidf.fit_transform(titles)

# 결과 크기 확인
print("도서 수:", tfidf_matrix.shape[0])
print("단어 수:", tfidf_matrix.shape[1])
print("TF-IDF 행렬 크기:", tfidf_matrix.shape)

도서 수: 199
단어 수: 536
TF-IDF 행렬 크기: (199, 536)


### 실행 결과 요약

- 도서 수 확인: TF-IDF 변환에 사용된 도서는 총 **199권**이다.
- 단어 수 확인: TfidfVectorizer가 만든 단어는 총 **536개**이다.
- 행렬 크기 확인: TF-IDF 행렬의 크기는 **(199, 536)**으로 나타났다.
- 행 의미 확인: 각 행은 하나의 도서를 의미한다.
- 열 의미 확인: 각 열은 도서 제목에서 추출된 하나의 단어를 의미한다.
- 결과 확인: 199권의 도서가 536개의 단어 기준으로 TF-IDF 벡터로 변환되어, 이후 도서 간 코사인 유사도를 계산할 수 있는 상태가 되었다.

## 실습7. 기준 도서 선택하기

In [6]:
# 기준 도서의 index 선택
selected_index = 0

# 선택한 index의 상품명 확인
selected_title = df_reco.loc[
    selected_index,
    "상품명"
]

print("선택 도서:", selected_title)

선택 도서: 소년이 온다


### 실행 결과 요약

- 기준 도서 확인: 추천의 기준 도서로 `소년이 온다`가 선택되었다.
- index 확인: `selected_index = 0`에 해당하는 도서가 `소년이 온다`로 정상적으로 연결되었다.
- 추천 기준 설정: 이후 `소년이 온다`의 TF-IDF 벡터와 다른 도서들의 벡터를 비교해 유사한 도서를 찾을 수 있다.
- 결과 확인: 추천 시스템에서 사용할 기준 도서가 정상적으로 선택되었다.

## 실습 8. 선택 도서와 전체 도서의 유사도 계산하기

In [7]:
# 선택한 도서의 TF-IDF 벡터 가져오기
selected_vector = tfidf_matrix[selected_index]

# 선택 도서와 전체 도서의 코사인 유사도 계산
similarity_scores = cosine_similarity(
    selected_vector,
    tfidf_matrix
).flatten()

# 결과 개수 확인
print("유사도 개수:", len(similarity_scores))
print("전체 도서 수:", len(df_reco))

# 선택 도서 자기 자신과의 유사도 확인
print("자기 자신과의 유사도:", similarity_scores[selected_index])

유사도 개수: 199
전체 도서 수: 199
자기 자신과의 유사도: 1.0000000000000002


### 실행 결과 요약

- 유사도 개수 확인: 계산된 코사인 유사도는 총 **199개**로 나타났다.
- 전체 도서 수 확인: 추천 대상 도서 수도 총 **199권**으로 확인되었다.
- 인덱스 대응 확인: 유사도 개수와 도서 수가 같아 각 유사도 점수가 원본 도서와 1:1로 대응함을 확인했다.
- 자기 자신과의 유사도 확인: `소년이 온다`와 자기 자신의 유사도는 약 **1.0**으로 나타났다.
- 소수점 오차 확인: `1.0000000000000002`는 부동소수점 계산에서 발생할 수 있는 아주 작은 오차로, 사실상 **1과 동일한 값**으로 해석할 수 있다.
- 결과 확인: 기준 도서와 전체 199권의 유사도가 정상적으로 계산되었고, 이후 자기 자신을 제외한 뒤 유사도가 높은 도서를 추천 후보로 선택할 수 있는 상태가 되었다.

## 실습 9. 유사도가 높은 순서 확인하기

In [9]:
# 도서 위치 번호, 상품명, 유사도 점수를 하나의 표로 정리
score_df = pd.DataFrame({
    "index": np.arange(len(df_reco)),
    "상품명": df_reco["상품명"].to_numpy(),
    "similarity": similarity_scores
})

# 유사도가 높은 순서대로 정렬
score_df = (
    score_df
    .sort_values("similarity", ascending=False)
    .reset_index(drop=True)
)

# 상위 10개 확인
score_df.head(10)

,index,상품명,similarity
0,0,소년이 온다,1.0
1,1,모순,0.0
2,2,결국 국민이 합니다,0.0
3,3,혼모노,0.0
4,4,급류,0.0
5,5,초역 부처의 말,0.0
6,6,청춘의 독서(특별증보판),0.0
7,7,어른의 행복은 조용하다,0.0
8,8,채식주의자,0.0
9,9,단 한 번의 삶(강물에디션 활판인쇄 한정판),0.0


### 실행 결과 요약

- 기준 도서 확인: `소년이 온다`를 기준으로 전체 도서와 코사인 유사도를 계산했다.
- 유사도 확인: 자기 자신을 제외한 다른 도서들의 유사도는 모두 `0`으로 나타났다.
- 원인 확인: `소년이`, `온다`라는 단어를 다른 도서 제목에서 공유하지 않아 공통 TF-IDF 특징이 없었다.
- 의미 확인: 현재 추천 방식은 제목에서 동일한 단어가 겹칠 때 유사도를 계산하는 구조이므로, 의미가 비슷하더라도 공통 단어가 없으면 유사도가 0이 될 수 있다.
- 한계 확인: 제목 단어만 사용하는 TF-IDF 기반 추천은 짧은 제목이나 고유한 표현을 가진 도서에서는 추천 성능이 제한될 수 있다.

## 실습 10. 자기 자신을 제외하고 Top 5 만들기

In [12]:
# 유사도가 높은 순서대로 도서 index 정렬
sorted_indices = similarity_scores.argsort()[::-1]

# 자기 자신은 제외하고 상위 5개 선택
recommended_indices = [
    idx
    for idx in sorted_indices
    if idx != selected_index
][:5]

print("추천 index:", recommended_indices)

# 추천 도서와 유사도 확인
result = df_reco.iloc[recommended_indices][["상품명"]].copy()

result["similarity"] = [
    round(float(similarity_scores[idx]), 4)
    for idx in recommended_indices
]

result

추천 index: [np.int64(197), np.int64(198), np.int64(195), np.int64(194), np.int64(193)]


,상품명,similarity
197,당신에게 분명 좋은 일만 생길 거예요,0.0
198,결핍은 우리를 어떻게 변화시키는가,0.0
195,당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션),0.0
194,용의자 X의 헌신,0.0
193,미치도록 보고 싶었던 돈의 얼굴,0.0


### 실행 결과 요약

- 추천 index 확인: 자기 자신을 제외한 뒤 상위 5개의 도서 index가 선택되었다.
- 추천 결과 확인: `당신에게 분명 좋은 일만 생길 거예요`, `결핍은 우리를 어떻게 변화시키는가` 등 5권이 추천 후보로 출력되었다.
- 유사도 확인: 선택된 5권의 코사인 유사도는 모두 **0.0**으로 나타났다.
- 원인 확인: 기준 도서 `소년이 온다`와 추천 후보 제목 사이에 공통 TF-IDF 토큰이 없어 실제 텍스트 유사도가 계산되지 않았다.
- Top 5 의미 확인: 유사도가 모두 0인 상태에서 선택된 5권은 단순히 정렬 결과에서 남은 index일 뿐, 실제로 `소년이 온다`와 비슷한 도서라고 해석할 수 없다.
- 한계 확인: 제목 단어만 사용하는 TF-IDF 추천은 공통 단어가 없는 경우 의미적으로 비슷한 도서를 찾지 못할 수 있다.
- 결과 확인: 추천 로직 자체는 정상적으로 동작했지만, 현재 기준 도서에서는 의미 있는 추천 결과를 만들기 어렵다는 점을 확인했다.

## 실습 11. 추천 결과에 메타데이터 추가하기

 

제목만 보는 것보다 저자, 출판사, 분야를 함께 보면 추천 결과를 검토하기 쉽습니다.

In [13]:
# 추천 결과에 함께 보여줄 컬럼 선택
available_columns = [
    column
    for column in [
        "상품명",
        "저자",
        "출판사",
        "분야",
    ]
    if column in df_reco.columns
]

# 추천 도서 정보 가져오기
result = df_reco.iloc[
    recommended_indices
][available_columns].copy()

# 유사도 추가
result["similarity"] = [
    round(float(similarity_scores[idx]), 4)
    for idx in recommended_indices
]

# 선택 도서와 추천 결과 확인
print("선택 도서:")
print(selected_title)

print("\n추천 도서:")
display(result)

선택 도서:
소년이 온다

추천 도서:


,상품명,저자,출판사,분야,similarity
197,당신에게 분명 좋은 일만 생길 거예요,이슬비,다담북스,시/에세이,0.0
198,결핍은 우리를 어떻게 변화시키는가,센딜 멀레이너선 외,빌리버튼,경제/경영,0.0
195,당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션),김상현,필름(Feelm),시/에세이,0.0
194,용의자 X의 헌신,히가시노 게이고,재인,소설,0.0
193,미치도록 보고 싶었던 돈의 얼굴,EBS 돈의 얼굴 제작진 외,영진닷컴,경제/경영,0.0


### 실행 결과 요약

- 선택 도서 확인: 기준 도서로 `소년이 온다`가 선택되었다.
- 추천 정보 확장: 추천 결과에 `상품명`, `저자`, `출판사`, `분야`, `similarity`를 함께 표시했다.
- 추천 후보 확인: 시/에세이, 경제/경영, 소설 등 여러 분야의 도서가 함께 출력되었다.
- 유사도 확인: 추천된 5권의 코사인 유사도는 모두 `0.0`으로 나타났다.
- 분야 비교: 기준 도서 `소년이 온다`는 소설이지만, 추천 결과에는 소설 외에도 시/에세이와 경제/경영 도서가 포함되었다.
- 해석 주의: 현재 추천된 도서들은 제목에서 공통 TF-IDF 단어가 없어 실제로 유사한 책이라고 보기 어렵다.
- 결과 확인: 메타데이터를 함께 확인하면서 단순히 추천 결과만 보는 것이 아니라, 분야와 저자 등의 정보를 이용해 추천의 타당성을 직접 검토할 수 있었다.

## 실습 12. 추천 함수 만들기

In [17]:
def recommend_books(
    selected_index,
    df,
    tfidf_matrix,
    top_n=5,
):
    # index 범위 확인
    if selected_index < 0 or selected_index >= len(df):
        raise IndexError(
            f"유효하지 않은 index입니다: {selected_index}"
        )

    # 선택한 도서 제목
    selected_title = df.iloc[selected_index]["상품명"]

    # 선택 도서와 전체 도서의 코사인 유사도 계산
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix,
    ).flatten()

    # 유사도가 높은 순서대로 index 정렬
    sorted_indices = similarity_scores.argsort()[::-1]

    recommended_indices = []

    for idx in sorted_indices:

        # 자기 자신 제외
        if idx == selected_index:
            continue

        # 동일한 제목 제외
        if df.iloc[idx]["상품명"] == selected_title:
            continue

        recommended_indices.append(idx)

        # Top N까지 모으면 종료
        if len(recommended_indices) >= top_n:
            break

    # 출력할 컬럼 선택
    columns = [
        column
        for column in [
            "상품명",
            "저자",
            "출판사",
            "분야",
        ]
        if column in df.columns
    ]

    # 추천 도서 정보 가져오기
    result = df.iloc[
        recommended_indices
    ][columns].copy()

    # 유사도 점수 추가
    result["similarity"] = [
        round(float(similarity_scores[idx]), 4)
        for idx in recommended_indices
    ]

    return result.reset_index(drop=True)

In [18]:
recommend_books(
    selected_index=0,
    df=df_reco,
    tfidf_matrix=tfidf_matrix,
    top_n=5
)

,상품명,저자,출판사,분야,similarity
0,당신에게 분명 좋은 일만 생길 거예요,이슬비,다담북스,시/에세이,0.0
1,결핍은 우리를 어떻게 변화시키는가,센딜 멀레이너선 외,빌리버튼,경제/경영,0.0
2,당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션),김상현,필름(Feelm),시/에세이,0.0
3,용의자 X의 헌신,히가시노 게이고,재인,소설,0.0
4,미치도록 보고 싶었던 돈의 얼굴,EBS 돈의 얼굴 제작진 외,영진닷컴,경제/경영,0.0


### 실행 결과 요약

- 추천 함수 실행: `recommend_books()` 함수를 사용해 기준 도서 `소년이 온다`의 추천 결과를 생성했다.
- 추천 개수 확인: 자기 자신과 동일 제목을 제외하고 총 **5권**이 반환되었다.
- 메타데이터 확인: 추천 결과에 `상품명`, `저자`, `출판사`, `분야`, `similarity`가 함께 표시되었다.
- 유사도 확인: 추천된 5권의 코사인 유사도는 모두 **0.0**으로 나타났다.
- 결과 해석: 함수 자체는 정상적으로 동작했지만, 기준 도서와 공통 TF-IDF 단어가 없어 의미 있는 유사 추천은 이루어지지 않았다.
- 한계 확인: 현재 방식은 제목의 공통 단어를 기반으로 하기 때문에, 의미적으로 비슷한 도서라도 같은 단어가 없으면 유사도가 0이 될 수 있다.
- 결과 확인: 추천 로직을 함수로 묶는 데 성공했고, 다른 도서를 기준으로 선택하면 같은 함수를 반복해서 사용할 수 있다.

## 14. 추천 결과 저장하기

In [19]:
# 추천 결과 만들기
recommendations = recommend_books(
    selected_index=0,
    df=df_reco,
    tfidf_matrix=tfidf_matrix,
    top_n=5
)

# CSV 파일로 저장
recommendations.to_csv(
    "chapter05_recommendations.csv",
    index=False,
    encoding="utf-8-sig"
)

# 다시 불러와 저장 결과 확인
saved_recommendations = pd.read_csv(
    "chapter05_recommendations.csv",
    encoding="utf-8-sig"
)

saved_recommendations

,상품명,저자,출판사,분야,similarity
0,당신에게 분명 좋은 일만 생길 거예요,이슬비,다담북스,시/에세이,0.0
1,결핍은 우리를 어떻게 변화시키는가,센딜 멀레이너선 외,빌리버튼,경제/경영,0.0
2,당신은 결국 무엇이든 해내는 사람(특별 리커버 에디션),김상현,필름(Feelm),시/에세이,0.0
3,용의자 X의 헌신,히가시노 게이고,재인,소설,0.0
4,미치도록 보고 싶었던 돈의 얼굴,EBS 돈의 얼굴 제작진 외,영진닷컴,경제/경영,0.0


### 실행 결과 요약

- 파일 재확인: 저장한 `chapter05_recommendations.csv` 파일을 다시 불러와 결과를 확인했다.
- 추천 개수 확인: 총 5권의 추천 결과가 정상적으로 저장되어 있다.
- 컬럼 확인: `상품명`, `저자`, `출판사`, `분야`, `similarity` 정보가 그대로 유지되었다.
- 한글 확인: 상품명과 저자, 출판사, 분야가 한글 깨짐 없이 정상적으로 저장되었다.
- 유사도 확인: 현재 기준 도서 `소년이 온다`와 추천된 도서들의 유사도는 모두 `0.0`으로 저장되었다.
- 결과 확인: 추천 결과 생성 → CSV 저장 → 다시 불러오기까지 전체 과정이 정상적으로 완료되었다.